## nb28 — Author Paper Panel Collection

For each author (603 award + 603 control from matched_pairs_clean.csv),
fetch all their papers in the window [award_year-5, award_year+5] from OpenAlex.

Output: `data/cd_trajectory/author_papers_panel.csv`
Columns: author_id, paper_id, paper_year, award_year, group, relative_year

In [1]:
import pandas as pd
import requests
import time
from pathlib import Path

ROOT    = Path('..')
MATCHED = ROOT / 'data' / 'matched'
OUT_DIR = ROOT / 'data' / 'cd_trajectory'
OUT_DIR.mkdir(parents=True, exist_ok=True)

API_KEY = 'A08hCjeUoeVKA9toVsfCpF'
WINDOW  = 5

In [2]:
pairs = pd.read_csv(MATCHED / 'matched_pairs_clean.csv')
print(pairs.shape)
print(pairs.columns.tolist())
pairs.head(3)

(603, 10)
['treated_id', 'treated_name', 'control_id', 'control_name', 'conference', 'award_year', 'treat_pos', 'ctrl_pos', 'match_dist', 'core_rank']


,treated_id,treated_name,control_id,control_name,conference,award_year,treat_pos,ctrl_pos,match_dist,core_rank
0,https://openalex.org/A5014823249,Jincheng Mei,https://openalex.org/A5026521600,Chenjun Xiao,AAAI,2018,2,1,0.0,A*
1,https://openalex.org/A5069646529,Adhiguna Kuncoro,https://openalex.org/A5111222692,Chris Dyer,ACL,2018,3,2,1.0,Unranked
2,https://openalex.org/A5032195981,Sang-Gyun An,https://openalex.org/A5010677076,Martin Müller,CHI,2018,2,3,0.0,A*


In [3]:
treated_list = pairs[['treated_id', 'award_year']].rename(columns={'treated_id': 'author_id'})
treated_list['group'] = 'award'

control_list = pairs[['control_id', 'award_year']].rename(columns={'control_id': 'author_id'})
control_list['group'] = 'control'

authors = pd.concat([treated_list, control_list], ignore_index=True)
authors = authors.drop_duplicates(subset='author_id').reset_index(drop=True)

print(f'Total unique authors: {len(authors)}')
print(authors['group'].value_counts())

Total unique authors: 796
group
award      603
control    193
Name: count, dtype: int64


In [4]:
def fetch_author_papers(author_id, year_min, year_max):
    aid = author_id.split('/')[-1]
    url = 'https://api.openalex.org/works'
    papers = []
    cursor = '*'
    while True:
        params = {
            'filter': f'author.id:{aid},publication_year:{year_min}-{year_max}',
            'per-page': 200,
            'select': 'id,publication_year',
            'cursor': cursor,
            'api_key': API_KEY,
        }
        r = requests.get(url, params=params, timeout=15)
        if r.status_code != 200:
            break
        data = r.json()
        results = data.get('results', [])
        for w in results:
            papers.append({'paper_id': w['id'].split('/')[-1], 'paper_year': w['publication_year']})
        cursor = data.get('meta', {}).get('next_cursor')
        if not cursor or not results:
            break
        time.sleep(0.15)
    return papers

In [5]:
CHECKPOINT = OUT_DIR / 'author_papers_panel.csv'

if CHECKPOINT.exists():
    done_df  = pd.read_csv(CHECKPOINT)
    done_ids = set(done_df['author_id'].unique())
    print(f'Resuming — {len(done_ids)} authors already done')
else:
    done_df  = pd.DataFrame()
    done_ids = set()

rows = []
todo = authors[~authors['author_id'].isin(done_ids)].reset_index(drop=True)
print(f'Remaining: {len(todo)} authors')

for i, row in todo.iterrows():
    aid        = row['author_id']
    award_year = int(row['award_year'])
    group      = row['group']
    yr_min     = award_year - WINDOW
    yr_max     = award_year + WINDOW

    papers = fetch_author_papers(aid, yr_min, yr_max)
    for p in papers:
        rows.append({
            'author_id':     aid,
            'group':         group,
            'award_year':    award_year,
            'paper_id':      p['paper_id'],
            'paper_year':    p['paper_year'],
            'relative_year': p['paper_year'] - award_year,
        })

    if (i + 1) % 50 == 0 or (i + 1) == len(todo):
        batch_df = pd.DataFrame(rows)
        combined = pd.concat([done_df, batch_df], ignore_index=True)
        combined.to_csv(CHECKPOINT, index=False)
        done_df  = combined
        rows     = []
        print(f'[{i+1}/{len(todo)}] Saved — total rows: {len(done_df)}')

    time.sleep(0.2)

print('Done!')
print(done_df.groupby('group')['author_id'].nunique())

Remaining: 796 authors
[50/796] Saved — total rows: 1249
[100/796] Saved — total rows: 2510
[150/796] Saved — total rows: 3658
[200/796] Saved — total rows: 5215
[250/796] Saved — total rows: 6025
[300/796] Saved — total rows: 7371
[350/796] Saved — total rows: 8749
[400/796] Saved — total rows: 9955
[450/796] Saved — total rows: 11219
[500/796] Saved — total rows: 12403
[550/796] Saved — total rows: 13523
[600/796] Saved — total rows: 14683
[650/796] Saved — total rows: 21271
[700/796] Saved — total rows: 24465
[750/796] Saved — total rows: 28965
[796/796] Saved — total rows: 30759
Done!
group
award      602
control    188
Name: author_id, dtype: int64


In [6]:
panel = pd.read_csv(CHECKPOINT)
print(panel.shape)
print(panel.groupby('group')['author_id'].nunique())
print(panel.groupby('group')['paper_id'].nunique())
panel.head()

(30759, 6)
group
award      602
control    188
Name: author_id, dtype: int64
group
award      14100
control    14966
Name: paper_id, dtype: int64


,author_id,group,award_year,paper_id,paper_year,relative_year
0,https://openalex.org/A5014823249,award,2018,W2407964224,2014,-4
1,https://openalex.org/A5014823249,award,2018,W2788603415,2018,0
2,https://openalex.org/A5014823249,award,2018,W2759170062,2017,-1
3,https://openalex.org/A5014823249,award,2018,W2965050397,2019,1
4,https://openalex.org/A5014823249,award,2018,W3098410977,2020,2
